## Load Libraries

In [ ]:
from pathlib import Path
import pymongo
import os
from bson import ObjectId
import json
import copy

print("Libraries loaded successfully")

OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)

print("Output directory created")

Libraries loaded successfully
Output directory created


## Connect to DB

In [ ]:
domain = os.environ.get('MONGO_DOMAIN', 'localhost')
port = os.environ.get('MONGO_PORT', 27017)
uname = os.environ.get('MONGO_USERNAME', 'root')
password = os.environ.get('MONGO_PASSWORD', 'example')

client = pymongo.MongoClient(f'mongodb://{uname}:{password}@{domain}:{port}/')

# User survey database and collection
db_survey_user = client['user_survey_clean_data']
survey_user_valid_ids_collection = db_survey_user['cleaned_user_survey_ids']
survey_user_valid_ids = [doc['survey_id'] for doc in survey_user_valid_ids_collection.find({})]
survey_user_answer_collection = db_survey_user['cleaned_user_survey_responses']

# Expert survey database and collection
db_survey_expert = client['expert_survey_clean_data']
survey_expert_cybersecurity_valid_ids_collection = db_survey_expert['cleaned_expert_survey_cybersecurity_ids']
survey_expert_cybersecurity_valid_ids = [doc['survey_id'] for doc in survey_expert_cybersecurity_valid_ids_collection.find({})]
survey_expert_policy_valid_ids_collection = db_survey_expert['cleaned_expert_survey_policy_ids']
survey_expert_policy_valid_ids = [doc['survey_id'] for doc in survey_expert_policy_valid_ids_collection.find({})]
survey_expert_law_valid_ids_collection = db_survey_expert['cleaned_expert_survey_law_ids']
survey_expert_law_valid_ids = [doc['survey_id'] for doc in survey_expert_law_valid_ids_collection.find({})]
survey_expert_ethics_valid_ids_collection = db_survey_expert['cleaned_expert_survey_ethics_ids']
survey_expert_ethics_valid_ids = [doc['survey_id'] for doc in survey_expert_ethics_valid_ids_collection.find({})]
survey_expert_answer_collection = db_survey_expert['cleaned_expert_survey_responses']

# User study database
db_user_study = client['user_study_clean_data']
user_study_collection = db_user_study['cleaned_user_study_ids']
user_study_valid_ids = valid_ids_from_db = [doc['study_id'] for doc in user_study_collection.find({})]
user_study_answer_collection = db_user_study['cleaned_user_study_responses']

## Survey load website and credential data

In [41]:
# Load website names and categorization as well as credential names from JSON files

file_path = "files/website_data.json"
with open(file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

website_categorization = {}
website_names = []
categories = []
sizes = ["Small", "Medium", "Large"]
hqs = ["China", "USA", "Europe"]
categories_with_different_countries = []
credential_categorization = {}
credentials = []

for category, websites in json_data.items():
    for site in websites:
        name = site["name"]
        cat = site.get("category", category)
        hq = site.get("hq", "")
        size = site.get("size", "")
        website_categorization[name] = (cat, hq, size)
        if not cat in categories:
            categories.append(cat)
        website_names.append(name)
        if hq == "China" and not cat in categories_with_different_countries:
            categories_with_different_countries.append(cat)

file_path = "files/credential_data.json"
with open(file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

for credential in json_data:
    name = credential["name"]
    id = credential["file_name"]
    credentials.append(name)
    credential_categorization[id] = name

print(website_categorization)
print(website_names)
print(credential_categorization)
print(credentials)
print(categories)
print(sizes)
print(hqs)
print(categories_with_different_countries)

{'DocMorris': ('Pharmacy', 'Switzerland', 'Large'), 'Redcare Pharmacy': ('Pharmacy', 'The Netherlands', 'Large'), 'NewPharma': ('Pharmacy', 'Belgium', 'Large'), 'Apotea': ('Pharmacy', 'Sweden', 'Medium'), 'Pharmacy2U': ('Pharmacy', 'UK', 'Medium'), 'Apteka Gemini': ('Pharmacy', 'Poland', 'Medium'), 'Farmacias': ('Pharmacy', 'Spain', 'Small'), 'Farmacia Loreto Gallo': ('Pharmacy', 'Italy', 'Small'), 'Euro-Pharmas': ('Pharmacy', 'France', 'Small'), 'Homedoctor': ('Online Doctors', 'Spain', 'Small'), 'Qare': ('Online Doctors', 'France', 'Small'), 'Kry': ('Online Doctors', 'Sweden', 'Small'), 'Teleclinic': ('Online Doctors', 'Germany', 'Small'), 'SoS Pediatra': ('Online Doctors', 'Italy', 'Small'), 'Medgate': ('Online Doctors', 'Switzerland', 'Small'), 'Official Government Website of Germany': ('Government', 'Germany', 'Large'), 'Official Government Website of France': ('Government', 'France', 'Large'), 'Official Government Website of Italy': ('Government', 'Italy', 'Small'), 'Official Gov

## User Survey: Generate comfort_frequency_breakdown data

In [42]:
number_of_responses_per_credential = {name: 0 for name in credentials}
number_of_responses_per_credential_still_share = {name: 0 for name in credentials}

for id in survey_user_valid_ids:
    answers_credentials = survey_user_answer_collection.find_one({
        "id": id,
        "form": "credentialTasks"})
    if not answers_credentials:
        print(f"Error this ID should not be valid {id}")
        continue
    for answer in answers_credentials['values']:
        credential = answer["credential"]
        credential_name = credential_categorization.get(credential, None)
        if credential_name in number_of_responses_per_credential:
            number_of_responses_per_credential[credential_name] += 1
        else:
            print(f"Error credential {credential_name} not found in the list of credentials")
        if credential_name in number_of_responses_per_credential_still_share:
            if answer["discomfort"] is not None:
                number_of_responses_per_credential_still_share[credential_name] += 1
        else:
            print(f"Error credential {credential_name} not found in the list of credentials")
print("Number of responses per credential:")
for credential, count in number_of_responses_per_credential.items():
    print(f"- {credential}: {count}")

Number of responses per credential:
- Official Identity Document: 1012
- Driver's license: 866
- Birth certificate: 876
- Marriage certificate: 262
- Visa: 330
- Bank credentials: 752
- University transcript: 628
- Diploma: 720
- Employment history: 494
- Professional licenses: 214
- Health insurance: 921
- Prescriptions: 733
- Medical records: 566
- Disability status: 51


In [43]:
average_frequency_for_credentials = {name: 0 for name in credentials}
average_comfort_for_credentials = {name: 0 for name in credentials}
average_still_share_for_credentials = {name: 0 for name in credentials}

for id in survey_user_valid_ids:
    answers_credentials = survey_user_answer_collection.find_one({
        "id": id,
        "form": "credentialTasks"})
    if not answers_credentials:
        print(f"Error this ID should not be valid {id}")
        continue
    for answer in answers_credentials['values']:
        credential = answer["credential"]
        credential_name = credential_categorization.get(credential, None)
        if credential_name in average_frequency_for_credentials:
            average_frequency_for_credentials[credential_name] += answer['frequency']
        else:
            print(f"Error credential {credential_name} not found in the list of credentials")
        if credential_name in average_comfort_for_credentials:
            average_comfort_for_credentials[credential_name] += answer['comfort']
        else:
            print(f"Error credential {credential_name} not found in the list of credentials")
        if credential_name in average_still_share_for_credentials:
            if answer['discomfort'] is not None:
                average_still_share_for_credentials[credential_name] += answer['discomfort']
        else:
            print(f"Error credential {credential_name} not found in the list of credentials")

for credential in average_frequency_for_credentials:
    if number_of_responses_per_credential[credential] > 0:
        average_frequency_for_credentials[credential] /= number_of_responses_per_credential[credential]
        average_comfort_for_credentials[credential] /= number_of_responses_per_credential[credential]
    if number_of_responses_per_credential_still_share[credential] > 0:
        average_still_share_for_credentials[credential] /= number_of_responses_per_credential_still_share[credential]

print("Average frequency per credential:")
for credential, count in average_frequency_for_credentials.items():
    print(f"- {credential}: {count}")
print("Average comfort per credential:")
for credential, count in average_comfort_for_credentials.items():
    print(f"- {credential}: {count}")
print("Average still share per credential:")
for credential, count in average_still_share_for_credentials.items():
    print(f"- {credential}: {count}")

output_file = "data/comfort_frequency_breakdown.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Comfort breakdown: \n")
    for credential in credentials:
        f.write(f"- {credential}:\n")
        f.write(f"  - Average frequency: {average_frequency_for_credentials[credential]} ({number_of_responses_per_credential[credential]})\n")
        f.write(f"  - Average comfort: {average_comfort_for_credentials[credential]} ({number_of_responses_per_credential[credential]})\n")
        f.write(f"  - Average still share: {average_still_share_for_credentials[credential]} ({number_of_responses_per_credential_still_share[credential]})\n")

Average frequency per credential:
- Official Identity Document: 2.4594861660079053
- Driver's license: 3.4191685912240186
- Birth certificate: 4.416666666666667
- Marriage certificate: 4.3320610687022905
- Visa: 3.4484848484848483
- Bank credentials: 3.4388297872340425
- University transcript: 3.855095541401274
- Diploma: 4.0472222222222225
- Employment history: 3.5303643724696356
- Professional licenses: 3.696261682242991
- Health insurance: 2.43213897937025
- Prescriptions: 2.622100954979536
- Medical records: 3.8180212014134276
- Disability status: 3.196078431372549
Average comfort per credential:
- Official Identity Document: 3.58399209486166
- Driver's license: 3.3787528868360277
- Birth certificate: 3.9189497716894977
- Marriage certificate: 3.816793893129771
- Visa: 3.603030303030303
- Bank credentials: 4.6063829787234045
- University transcript: 2.9808917197452227
- Diploma: 2.7666666666666666
- Employment history: 3.08502024291498
- Professional licenses: 3.02803738317757
- He

## User survey: Get number of users giving a credential to a website

In [44]:
number_of_responses_per_website = {name: 0 for name in website_names}

for id in survey_user_valid_ids:
    answers_website = survey_user_answer_collection.find_one({
        "id": id,
        "form": "websiteCredentialsOpinions"})
    if not answers_website:
        print(f"Error this ID should not be valid {id}")
        continue
    for answer in answers_website['values']:
        website = answer["website"]
        if website in number_of_responses_per_website:
            number_of_responses_per_website[website] += 1
        else:
            print(f"Error website {website} not found in the list of websites")
print("Number of responses per website:")
for website, count in number_of_responses_per_website.items():
    print(f"- {website}: {count}")


Number of responses per website:
- DocMorris: 140
- Redcare Pharmacy: 128
- NewPharma: 126
- Apotea: 115
- Pharmacy2U: 149
- Apteka Gemini: 141
- Farmacias: 123
- Farmacia Loreto Gallo: 125
- Euro-Pharmas: 141
- Homedoctor: 184
- Qare: 188
- Kry: 177
- Teleclinic: 216
- SoS Pediatra: 178
- Medgate: 200
- Official Government Website of Germany: 152
- Official Government Website of France: 137
- Official Government Website of Italy: 126
- Official Government Website of the UK: 139
- Official Government Website of Denmark: 141
- Official Government Website of the Netherlands: 116
- Official Government Website of Poland: 134
- Official Government Website of Lithuania: 139
- Official Government Website of Estonia: 137
- Flixbus: 124
- Blablacar: 130
- Trainline: 143
- Trenitalia: 128
- Eurail: 146
- Alsa: 133
- Ecolines: 121
- RegioJet: 139
- Faniani Coaches: 143
- HSBC: 150
- ABN Amro: 124
- Sparkasse: 129
- Sella: 129
- AIB Group: 122
- Handelsbanken: 126
- Bank of Valletta: 138
- Cecaban

In [45]:
# Compute the absolute number of users who gave a credential to a specific website
number_users_giving_a_credential_to_website = {
    website: {credential: 0 for credential in credentials}
    for website in website_names
}

for id in survey_user_valid_ids:
    answers_website = survey_user_answer_collection.find_one({
        "id": id,
        "form": "websiteCredentialsOpinions"})
    if not answers_website:
        print(f"Error this ID should not be valid {id}")
        continue
    for answer in answers_website['values']:
        necessary = answer["necessaryCredentials"]
        permissible = answer["permissibleCredentials"]
        if necessary or permissible:
            combined = list(set(necessary + permissible))
            for credential_id in combined:
                if credential_id == "none":
                    continue
                credential_name = credential_categorization.get(credential_id, None)
                if not credential_name:
                    print(f"Error credential {credential_id} not found in the list of credentials")
                    continue
                website = answer["website"]
                if website in number_users_giving_a_credential_to_website:
                    number_users_giving_a_credential_to_website[website][credential_name] += 1
                else:
                    print(f"Error website {website} not found in the list of websites")

print("Number of users giving a credential to a specific website:")
for website, credentials_dict in number_users_giving_a_credential_to_website.items():
    print(f"- {website}:")
    for credential, count in credentials_dict.items():
        print(f"  - {credential}: {count}")

Number of users giving a credential to a specific website:
- DocMorris:
  - Official Identity Document: 64
  - Driver's license: 3
  - Birth certificate: 8
  - Marriage certificate: 2
  - Visa: 2
  - Bank credentials: 23
  - University transcript: 2
  - Diploma: 3
  - Employment history: 3
  - Professional licenses: 4
  - Health insurance: 105
  - Prescriptions: 120
  - Medical records: 63
  - Disability status: 30
- Redcare Pharmacy:
  - Official Identity Document: 65
  - Driver's license: 2
  - Birth certificate: 6
  - Marriage certificate: 2
  - Visa: 4
  - Bank credentials: 25
  - University transcript: 2
  - Diploma: 2
  - Employment history: 4
  - Professional licenses: 4
  - Health insurance: 89
  - Prescriptions: 111
  - Medical records: 64
  - Disability status: 39
- NewPharma:
  - Official Identity Document: 55
  - Driver's license: 4
  - Birth certificate: 3
  - Marriage certificate: 0
  - Visa: 1
  - Bank credentials: 12
  - University transcript: 0
  - Diploma: 1
  - Emplo

In [46]:
# Compute the percentage number of users who gave a credential to a specific website
percentage_users_giving_a_credential_to_website = copy.deepcopy(number_users_giving_a_credential_to_website)
for website, credentials_dict in percentage_users_giving_a_credential_to_website.items():
    total_responses = number_of_responses_per_website[website]
    for credential, count in credentials_dict.items():
        if total_responses > 0:
            percentage = (count / total_responses) * 100
        else:
            percentage = 0
        percentage_users_giving_a_credential_to_website[website][credential] = percentage

print("Percentage of users giving a credential to a specific website:")
for website, credentials_dict in percentage_users_giving_a_credential_to_website.items():
    print(f"- {website}:")
    for credential, percentage in credentials_dict.items():
        print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_website[website][credential]} / {number_of_responses_per_website[website]})")

output_file = "data/credential_percentage_per_website.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Percentage of users giving a credential to a specific website:\n")
    for website, credentials_dict in percentage_users_giving_a_credential_to_website.items():
        f.write(f"- {website}:\n")
        for credential, percentage in credentials_dict.items():
            f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_website[website][credential]} / {number_of_responses_per_website[website]})\n")

Percentage of users giving a credential to a specific website:
- DocMorris:
  - Official Identity Document: 45.71% (64 / 140)
  - Driver's license: 2.14% (3 / 140)
  - Birth certificate: 5.71% (8 / 140)
  - Marriage certificate: 1.43% (2 / 140)
  - Visa: 1.43% (2 / 140)
  - Bank credentials: 16.43% (23 / 140)
  - University transcript: 1.43% (2 / 140)
  - Diploma: 2.14% (3 / 140)
  - Employment history: 2.14% (3 / 140)
  - Professional licenses: 2.86% (4 / 140)
  - Health insurance: 75.00% (105 / 140)
  - Prescriptions: 85.71% (120 / 140)
  - Medical records: 45.00% (63 / 140)
  - Disability status: 21.43% (30 / 140)
- Redcare Pharmacy:
  - Official Identity Document: 50.78% (65 / 128)
  - Driver's license: 1.56% (2 / 128)
  - Birth certificate: 4.69% (6 / 128)
  - Marriage certificate: 1.56% (2 / 128)
  - Visa: 3.12% (4 / 128)
  - Bank credentials: 19.53% (25 / 128)
  - University transcript: 1.56% (2 / 128)
  - Diploma: 1.56% (2 / 128)
  - Employment history: 3.12% (4 / 128)
  - Prof

## User Survey: Get number of users giving a credential to a category

In [47]:
#Compute number of responses per category
number_of_responses_per_category = {category: 0 for category in categories}

for name, count in number_of_responses_per_website.items():
    category, hq, size = website_categorization.get(name, (None, None, None))
    if category:
        number_of_responses_per_category[category] += count
    else:
        print(f"Error website {name} not found in the list of categories")

print("Number of responses per category:")
for category, count in number_of_responses_per_category.items():
    print(f"- {category}: {count}")

Number of responses per category:
- Pharmacy: 1188
- Online Doctors: 1143
- Government: 1221
- International Ground Travel: 1207
- Banks: 1177
- Real Estate: 1145
- University: 1206
- News: 1201
- Car Rentals: 1249
- Video Games: 1147
- Social Media: 1148
- Encyclopedia: 1202
- Search Engine: 1181
- Payment Services: 1158
- Job Platforms: 1358
- Air Travel: 1397
- E-Commerce: 1372


In [48]:
#Compute number of responses per category with a specific size
number_of_responses_per_category_and_size = {
    (category, size): 0 for category in categories for size in sizes
}

for name, count in number_of_responses_per_website.items():
    category, hq, size = website_categorization.get(name, (None, None, None))
    if category:
        number_of_responses_per_category_and_size[(category, size)] += count
    else:
        print(f"Error website {name} not found in the list of categories")

print("Number of responses per category:")
for (category, size), count in number_of_responses_per_category_and_size.items():
    print(f"- {category} ({size}): {count}")

Number of responses per category:
- Pharmacy (Small): 389
- Pharmacy (Medium): 405
- Pharmacy (Large): 394
- Online Doctors (Small): 1143
- Online Doctors (Medium): 0
- Online Doctors (Large): 0
- Government (Small): 406
- Government (Medium): 134
- Government (Large): 681
- International Ground Travel (Small): 403
- International Ground Travel (Medium): 407
- International Ground Travel (Large): 397
- Banks (Small): 397
- Banks (Medium): 377
- Banks (Large): 403
- Real Estate (Small): 0
- Real Estate (Medium): 0
- Real Estate (Large): 1145
- University (Small): 411
- University (Medium): 387
- University (Large): 408
- News (Small): 390
- News (Medium): 391
- News (Large): 420
- Car Rentals (Small): 621
- Car Rentals (Medium): 0
- Car Rentals (Large): 628
- Video Games (Small): 0
- Video Games (Medium): 0
- Video Games (Large): 1147
- Social Media (Small): 0
- Social Media (Medium): 0
- Social Media (Large): 1148
- Encyclopedia (Small): 395
- Encyclopedia (Medium): 397
- Encyclopedia 

In [49]:
#Compute number of responses per category with a specific hq
number_of_responses_per_category_and_hq = {
    (category, hq): 0 for category in categories_with_different_countries for hq in hqs
}

for name, count in number_of_responses_per_website.items():
    category, hq, size = website_categorization.get(name, (None, None, None))
    if category:
        if not category in categories_with_different_countries:
            continue
        if hq == "China" or hq == "USA":
            number_of_responses_per_category_and_hq[(category, hq)] += count
        else:
            number_of_responses_per_category_and_hq[(category, "Europe")] += count
    else:
        print(f"Error website {name} not found in the list of categories")

print("Number of responses per category:")
for (category, hq), count in number_of_responses_per_category_and_hq.items():
    print(f"- {category} ({hq}): {count}")


Number of responses per category:
- Job Platforms (China): 471
- Job Platforms (USA): 444
- Job Platforms (Europe): 443
- Air Travel (China): 453
- Air Travel (USA): 465
- Air Travel (Europe): 479
- E-Commerce (China): 444
- E-Commerce (USA): 450
- E-Commerce (Europe): 478


In [50]:
# Compute the absolute number of users who gave a credential for each category

number_users_giving_a_credential_to_category = {
    category: {credential: 0 for credential in credentials}
    for category in categories
}

for website, credentials_dict in number_users_giving_a_credential_to_website.items():
    cat, hq, size = website_categorization.get(website, (None, None, None))
    if not cat:
        print(f"Error website {website} not found in the categorization")
        continue
    for credential, count in credentials_dict.items():
        number_users_giving_a_credential_to_category[cat][credential] += count
    
print("Number of users giving a credential to a specific category:")
for category, credentials_dict in number_users_giving_a_credential_to_category.items():
    print(f"- {category}:")
    for credential, count in credentials_dict.items():
        print(f"  - {credential}: {count}")

Number of users giving a credential to a specific category:
- Pharmacy:
  - Official Identity Document: 565
  - Driver's license: 30
  - Birth certificate: 34
  - Marriage certificate: 9
  - Visa: 14
  - Bank credentials: 194
  - University transcript: 7
  - Diploma: 12
  - Employment history: 12
  - Professional licenses: 25
  - Health insurance: 863
  - Prescriptions: 982
  - Medical records: 505
  - Disability status: 306
- Online Doctors:
  - Official Identity Document: 655
  - Driver's license: 25
  - Birth certificate: 87
  - Marriage certificate: 10
  - Visa: 21
  - Bank credentials: 76
  - University transcript: 8
  - Diploma: 15
  - Employment history: 31
  - Professional licenses: 19
  - Health insurance: 977
  - Prescriptions: 894
  - Medical records: 983
  - Disability status: 553
- Government:
  - Official Identity Document: 1172
  - Driver's license: 505
  - Birth certificate: 604
  - Marriage certificate: 483
  - Visa: 527
  - Bank credentials: 285
  - University transcr

In [51]:
# Compute the percentage of users who gave a credential for each category

percentage_users_giving_a_credential_to_category = copy.deepcopy(number_users_giving_a_credential_to_category)
for category, credentials_dict in percentage_users_giving_a_credential_to_category.items():
    total_responses = number_of_responses_per_category[category]
    for credential, count in credentials_dict.items():
        if total_responses > 0:
            percentage = (count / total_responses) * 100
        else:
            percentage = 0
        percentage_users_giving_a_credential_to_category[category][credential] = percentage

print("Percentage of users giving a credential to a specific category:")
for category, credentials_dict in percentage_users_giving_a_credential_to_category.items():
    print(f"- {category}:")
    for credential, percentage in credentials_dict.items():
        print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category[category][credential]} / {number_of_responses_per_category[category]})")

output_file = "data/user_survey_percentage_per_category.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Percentage of users giving a credential to a specific category:\n")
    for category, credentials_dict in percentage_users_giving_a_credential_to_category.items():
        f.write(f"- {category}:\n")
        for credential, percentage in credentials_dict.items():
            f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category[category][credential]} / {number_of_responses_per_category[category]})\n")

Percentage of users giving a credential to a specific category:
- Pharmacy:
  - Official Identity Document: 47.56% (565 / 1188)
  - Driver's license: 2.53% (30 / 1188)
  - Birth certificate: 2.86% (34 / 1188)
  - Marriage certificate: 0.76% (9 / 1188)
  - Visa: 1.18% (14 / 1188)
  - Bank credentials: 16.33% (194 / 1188)
  - University transcript: 0.59% (7 / 1188)
  - Diploma: 1.01% (12 / 1188)
  - Employment history: 1.01% (12 / 1188)
  - Professional licenses: 2.10% (25 / 1188)
  - Health insurance: 72.64% (863 / 1188)
  - Prescriptions: 82.66% (982 / 1188)
  - Medical records: 42.51% (505 / 1188)
  - Disability status: 25.76% (306 / 1188)
- Online Doctors:
  - Official Identity Document: 57.31% (655 / 1143)
  - Driver's license: 2.19% (25 / 1143)
  - Birth certificate: 7.61% (87 / 1143)
  - Marriage certificate: 0.87% (10 / 1143)
  - Visa: 1.84% (21 / 1143)
  - Bank credentials: 6.65% (76 / 1143)
  - University transcript: 0.70% (8 / 1143)
  - Diploma: 1.31% (15 / 1143)
  - Employmen

In [52]:
# Does the size of websites matter - Absolute number of users giving a credential to a category and specific size

number_users_giving_a_credential_to_category_and_size = {
    (category, size): {credential: 0 for credential in credentials}
    for category in categories for size in sizes
}

for website, credentials_dict in number_users_giving_a_credential_to_website.items():
    cat, hq, size = website_categorization.get(website, (None, None, None))
    if not cat:
        print(f"Error website {website} not found in the categorization")
        continue
    for credential, count in credentials_dict.items():
        number_users_giving_a_credential_to_category_and_size[(cat, size)][credential] += count

print("Number of users giving a credential to a specific category:")
for (category, size), credentials_dict in number_users_giving_a_credential_to_category_and_size.items():
    print(f"- {category} ({size}):")
    for credential, count in credentials_dict.items():
        print(f"  - {credential}: {count}")

Number of users giving a credential to a specific category:
- Pharmacy (Small):
  - Official Identity Document: 182
  - Driver's license: 7
  - Birth certificate: 7
  - Marriage certificate: 3
  - Visa: 4
  - Bank credentials: 67
  - University transcript: 1
  - Diploma: 5
  - Employment history: 3
  - Professional licenses: 8
  - Health insurance: 285
  - Prescriptions: 319
  - Medical records: 157
  - Disability status: 99
- Pharmacy (Medium):
  - Official Identity Document: 199
  - Driver's license: 14
  - Birth certificate: 10
  - Marriage certificate: 2
  - Visa: 3
  - Bank credentials: 67
  - University transcript: 2
  - Diploma: 1
  - Employment history: 2
  - Professional licenses: 7
  - Health insurance: 291
  - Prescriptions: 326
  - Medical records: 166
  - Disability status: 111
- Pharmacy (Large):
  - Official Identity Document: 184
  - Driver's license: 9
  - Birth certificate: 17
  - Marriage certificate: 4
  - Visa: 7
  - Bank credentials: 60
  - University transcript: 

In [53]:
# Does the size of websites matter - Percentage of users giving a credential to a category and specific size

percentage_users_giving_a_credential_to_category_and_size = copy.deepcopy(number_users_giving_a_credential_to_category_and_size)
for (category, size), credentials_dict in percentage_users_giving_a_credential_to_category_and_size.items():
    total_responses = number_of_responses_per_category_and_size[(category, size)]
    for credential, count in credentials_dict.items():
        if total_responses > 0:
            percentage = (count / total_responses) * 100
        else:
            percentage = 0
        percentage_users_giving_a_credential_to_category_and_size[(category, size)][credential] = percentage

print("Percentage of users giving a credential to a specific category:")
for (category, size), credentials_dict in percentage_users_giving_a_credential_to_category_and_size.items():
    print(f"- {category} ({size}):")
    for credential, percentage in credentials_dict.items():
        print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_and_size[(category, size)][credential]} / {number_of_responses_per_category_and_size[(category, size)]})")

output_file = "data/credential_percentage_per_category_and_size.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Percentage of users giving a credential to a specific category:\n")
    for (category, size), credentials_dict in percentage_users_giving_a_credential_to_category_and_size.items():
        f.write(f"- {category} ({size}):\n")
        for credential, percentage in credentials_dict.items():
            f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_and_size[(category, size)][credential]} / {number_of_responses_per_category_and_size[(category, size)]})\n")

Percentage of users giving a credential to a specific category:
- Pharmacy (Small):
  - Official Identity Document: 46.79% (182 / 389)
  - Driver's license: 1.80% (7 / 389)
  - Birth certificate: 1.80% (7 / 389)
  - Marriage certificate: 0.77% (3 / 389)
  - Visa: 1.03% (4 / 389)
  - Bank credentials: 17.22% (67 / 389)
  - University transcript: 0.26% (1 / 389)
  - Diploma: 1.29% (5 / 389)
  - Employment history: 0.77% (3 / 389)
  - Professional licenses: 2.06% (8 / 389)
  - Health insurance: 73.26% (285 / 389)
  - Prescriptions: 82.01% (319 / 389)
  - Medical records: 40.36% (157 / 389)
  - Disability status: 25.45% (99 / 389)
- Pharmacy (Medium):
  - Official Identity Document: 49.14% (199 / 405)
  - Driver's license: 3.46% (14 / 405)
  - Birth certificate: 2.47% (10 / 405)
  - Marriage certificate: 0.49% (2 / 405)
  - Visa: 0.74% (3 / 405)
  - Bank credentials: 16.54% (67 / 405)
  - University transcript: 0.49% (2 / 405)
  - Diploma: 0.25% (1 / 405)
  - Employment history: 0.49% (2 /

In [54]:
# Does website HQ matter - Absolute number of users giving a credential to a category and specific HQ
number_users_giving_a_credential_to_category_and_hq = {
    (category, hq): {credential: 0 for credential in credentials}
    for category in categories_with_different_countries for hq in hqs
}

for website, credentials_dict in number_users_giving_a_credential_to_website.items():
    cat, hq, size = website_categorization.get(website, (None, None, None))
    if not cat:
        print(f"Error website {website} not found in the categorization")
        continue
    for credential, count in credentials_dict.items():
        if not cat in categories_with_different_countries:
            continue
        if hq == "China" or hq == "USA":
            number_users_giving_a_credential_to_category_and_hq[(cat, hq)][credential] += count
        else:
            number_users_giving_a_credential_to_category_and_hq[(cat, "Europe")][credential] += count

print("Number of users giving a credential to a specific category:")
for (category, hq), credentials_dict in number_users_giving_a_credential_to_category_and_hq.items():
    print(f"- {category} ({hq}):")
    for credential, count in credentials_dict.items():
        print(f"  - {credential}: {count}")


Number of users giving a credential to a specific category:
- Job Platforms (China):
  - Official Identity Document: 238
  - Driver's license: 89
  - Birth certificate: 20
  - Marriage certificate: 5
  - Visa: 97
  - Bank credentials: 19
  - University transcript: 197
  - Diploma: 329
  - Employment history: 390
  - Professional licenses: 314
  - Health insurance: 33
  - Prescriptions: 5
  - Medical records: 18
  - Disability status: 78
- Job Platforms (USA):
  - Official Identity Document: 233
  - Driver's license: 72
  - Birth certificate: 11
  - Marriage certificate: 4
  - Visa: 57
  - Bank credentials: 12
  - University transcript: 149
  - Diploma: 301
  - Employment history: 371
  - Professional licenses: 318
  - Health insurance: 15
  - Prescriptions: 3
  - Medical records: 12
  - Disability status: 58
- Job Platforms (Europe):
  - Official Identity Document: 252
  - Driver's license: 97
  - Birth certificate: 17
  - Marriage certificate: 11
  - Visa: 69
  - Bank credentials: 12


In [55]:
# Does website HQ matter - Percentage number of users giving a credential to a category and specific HQ

percentage_users_giving_a_credential_to_category_and_hq = copy.deepcopy(number_users_giving_a_credential_to_category_and_hq)
for (category, hq), credentials_dict in percentage_users_giving_a_credential_to_category_and_hq.items():
    total_responses = number_of_responses_per_category_and_hq[(category, hq)]
    for credential, count in credentials_dict.items():
        if total_responses > 0:
            percentage = (count / total_responses) * 100
        else:
            percentage = 0
        percentage_users_giving_a_credential_to_category_and_hq[(category, hq)][credential] = percentage

print("Percentage of users giving a credential to a specific category:")
for (category, hq), credentials_dict in percentage_users_giving_a_credential_to_category_and_hq.items():
    print(f"- {category} ({hq}):")
    for credential, percentage in credentials_dict.items():
        print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_and_hq[(category, hq)][credential]} / {number_of_responses_per_category_and_hq[(category, hq)]})")

output_file = "data/credential_percentage_per_category_and_hq.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Percentage of users giving a credential to a specific category:\n")
    for (category, hq), credentials_dict in percentage_users_giving_a_credential_to_category_and_hq.items():
        f.write(f"- {category} ({hq}):\n")
        for credential, percentage in credentials_dict.items():
            f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_and_hq[(category, hq)][credential]} / {number_of_responses_per_category_and_hq[(category, hq)]})\n")

Percentage of users giving a credential to a specific category:
- Job Platforms (China):
  - Official Identity Document: 50.53% (238 / 471)
  - Driver's license: 18.90% (89 / 471)
  - Birth certificate: 4.25% (20 / 471)
  - Marriage certificate: 1.06% (5 / 471)
  - Visa: 20.59% (97 / 471)
  - Bank credentials: 4.03% (19 / 471)
  - University transcript: 41.83% (197 / 471)
  - Diploma: 69.85% (329 / 471)
  - Employment history: 82.80% (390 / 471)
  - Professional licenses: 66.67% (314 / 471)
  - Health insurance: 7.01% (33 / 471)
  - Prescriptions: 1.06% (5 / 471)
  - Medical records: 3.82% (18 / 471)
  - Disability status: 16.56% (78 / 471)
- Job Platforms (USA):
  - Official Identity Document: 52.48% (233 / 444)
  - Driver's license: 16.22% (72 / 444)
  - Birth certificate: 2.48% (11 / 444)
  - Marriage certificate: 0.90% (4 / 444)
  - Visa: 12.84% (57 / 444)
  - Bank credentials: 2.70% (12 / 444)
  - University transcript: 33.56% (149 / 444)
  - Diploma: 67.79% (301 / 444)
  - Employ

## User Survey: responses for User study scenarios

In [56]:
# Output same data as phase 2


with open("files/user_study_website_data.json", "r", encoding="utf-8") as f:
    scenario_to_website_data = json.load(f)

with open("files/user_study_scenario_id_mapping.json", "r", encoding="utf-8") as f:
    equivalent_scenarios = json.load(f)

map_id_to_scenario = {}
for scenario, data in equivalent_scenarios.items():
    for scenario_id in data['all_good'] + data['control'] + data['test']:
        map_id_to_scenario[scenario_id] = scenario

scenario_responses_percentage = {scenario: 0 for scenario in scenario_to_website_data.keys()}
scenario_total_responses = {scenario: 0 for scenario in scenario_to_website_data.keys()}

for scenario, data in scenario_to_website_data.items():
    website = data["website"]
    category = data["category_id"]
    credential = data["requested_credential_id"]
    if website == "uk_government":
        website = "Official Government Website of the UK"
    if website == "germany_government":
        website = "Official Government Website of Germany"
    if website == "france_government":
        website = "Official Government Website of France"
    if website not in percentage_users_giving_a_credential_to_website:
        print(f"Error {website} not found in percentage_users_giving_a_credential_to_website")
        continue
    credential_name = credential_categorization.get(credential, None)
    if not credential_name:
        print(f"Error credential {credential} not found in credential_categorization")
        continue
    total_number = number_of_responses_per_website.get(website, 0)
    percentage = percentage_users_giving_a_credential_to_website[website].get(credential_name, 0)
    scenario_responses_percentage[scenario] = percentage
    scenario_total_responses[scenario] = total_number

with open("data/survey_responses_for_user_study_scenarios.txt", "w", encoding="utf-8") as f:
    f.write("Scenario responses:\n")
    for scenario, percentage in scenario_responses_percentage.items():
        f.write(f"- {scenario}: {percentage:.2f}% ({scenario_total_responses[scenario]})\n")

print("Scenario responses:")
for scenario, percentage in scenario_responses_percentage.items():
    print(f"- {scenario}: {percentage:.2f}% ({scenario_total_responses[scenario]})")

Scenario responses:
- S1_1: 21.95% (82)
- S1_2: 21.84% (87)
- S1_3: 36.69% (139)
- S1_4: 11.22% (196)
- S1_5: 43.48% (207)
- S1_6: 86.44% (177)
- S1_7: 9.68% (124)
- S1_8: 40.13% (152)
- S1_9: 72.37% (152)
- S1_10: 83.80% (216)
- S2_1: 0.61% (165)
- S2_2: 8.53% (129)
- S2_3: 1.54% (130)
- S2_4: 2.31% (130)
- S2_5: 22.60% (146)
- S3_1: 2.27% (132)
- S3_2: 19.05% (147)
- S3_3: 0.00% (72)
- S3_4: 1.56% (128)
- S3_5: 0.00% (65)
- S3_6: 0.00% (70)
- S3_7: 3.54% (198)
- S3_8: 0.00% (85)
- S3_9: 2.67% (75)
- S3_10: 0.81% (124)
- S4_1: 85.50% (200)
- S4_2: 44.86% (185)
- S4_3: 84.21% (114)
- S4_4: 66.41% (131)
- S4_5: 75.00% (140)
- S5_1: 1.56% (128)
- S5_2: 3.93% (178)
- S5_3: 0.00% (133)
- S5_4: 1.69% (118)
- S5_5: 4.23% (71)
- S5_6: 0.68% (148)
- S5_7: 1.30% (77)
- S5_8: 0.92% (217)
- S5_9: 3.38% (148)
- S5_10: 4.46% (202)
- S6_1: 42.34% (137)
- S6_2: 48.95% (143)
- S6_3: 72.17% (115)
- S6_4: 76.15% (130)
- S6_5: 81.52% (184)
- S6_6: 86.70% (188)
- S6_7: 46.05% (76)
- S6_8: 90.91% (121)
- S

## Expert Survey: Load required auxiliary information

In [57]:
types = ["cybersecurity", "law", "policy", "ethics"]

file_path = "files/expert_website_data.json"
with open(file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

website_categorization = {}
website_names = []
categories = []
credential_categorization = {}
credentials = []

for code, websites in json_data.items():
    for site in websites:
        name = site["name"]
        cat = site["category"]
        website_categorization[name] = (cat)
        if not cat in categories:
            categories.append(cat)
        website_names.append(name)

file_path = "files/credential_data.json"
with open(file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

for credential in json_data:
    name = credential["name"]
    id = credential["file_name"]
    credentials.append(name)
    credential_categorization[id] = name

print(website_categorization)
print(website_names)
print(credential_categorization)
print(credentials)
print(categories)

{'Shop Apotheke': 'Pharmacy', 'Zava': 'Online Doctors', 'Official Government Website of Germany': 'Government', 'Stepstone': 'Job Portal', 'TU München': 'University', 'Lufthansa': 'Air Travel', 'FlixBus': 'International Ground Travel', 'Hertz': 'Car Rental', 'Sparkasse': 'Bank', 'ImmoScout24': 'Real Estate', 'Steam': 'Gaming', 'Amazon': 'E-commerce', 'Facebook': 'Social Media', 'Zeit': 'News', 'PayPal': 'Payment Services', 'DocMorris': 'Pharmacy', 'Qare': 'Online Doctors', 'Official Government Website of France': 'Government', 'Pole Emploi': 'Job Portal', 'Panthéon Sorbonne': 'University', 'Air France': 'Air Travel', 'Crédit Mutuel': 'Bank', 'SeLoger': 'Real Estate', 'Le Monde': 'News', 'Redcare': 'Pharmacy', 'SoS Pediatra': 'Online Doctors', 'Official Government Website of Italy': 'Government', 'Indeed': 'Job Portal', 'Bocconi': 'University', 'ITA Airways': 'Air Travel', 'Intesa Sanpaolo': 'Bank', 'Immobiliare': 'Real Estate', 'La Repubblica': 'News'}
['Shop Apotheke', 'Zava', 'Offici

## Expert Survey: Produce data about how many experts would give data to website

In [58]:
# Get number of responses per website and number of credentials given to website

number_of_responses_per_website_expert = {type: {name: 0 for name in website_names} for type in types}
number_users_giving_a_credential_to_website_necessary_expert = {
    type: {
        website: {credential: 0 for credential in credentials}
        for website in website_names}
    for type in types
}
number_users_giving_a_credential_to_website_permissible_expert = {
    type: {
        website: {credential: 0 for credential in credentials}
        for website in website_names}
    for type in types
}

for id in survey_expert_cybersecurity_valid_ids + survey_expert_law_valid_ids + survey_expert_policy_valid_ids + survey_expert_ethics_valid_ids:
    answers_website = survey_expert_answer_collection.find_one({
        "id": id,
        "form": "websiteCredentialsOpinions"})
    if not answers_website:
        print(f"Error this ID should not be valid {id}")
        continue
    for answer in answers_website['values']:
        website = answer["website"]
        if id in survey_expert_cybersecurity_valid_ids:
            type = "cybersecurity"
        elif id in survey_expert_law_valid_ids:
            type = "law"
        elif id in survey_expert_policy_valid_ids:
            type = "policy"
        elif id in survey_expert_ethics_valid_ids:
            type = "ethics"
        if website in number_of_responses_per_website_expert[type]:
            number_of_responses_per_website_expert[type][website] += 1
        else:
            print(f"Error website {website} not found in the list of websites")
        necessary = answer["necessaryCredentials"]
        permissible = answer["permissibleCredentials"]
        if necessary:
            for credential_id in necessary:
                if credential_id == "None":
                    continue
                if website in number_users_giving_a_credential_to_website_necessary_expert[type]:
                    number_users_giving_a_credential_to_website_necessary_expert[type][website][credential_id] += 1
                else:
                    print(f"Error website {website} not found in the list of websites")
        if permissible:
            for credential_id in permissible:
                if credential_id == "None":
                    continue
                if website in number_users_giving_a_credential_to_website_permissible_expert[type]:
                    number_users_giving_a_credential_to_website_permissible_expert[type][website][credential_id] += 1
                else:
                    print(f"Error website {website} not found in the list of websites")
                    
print("Number of expert responses per website:")
for type in types:
    print(f"Type: {type}")
    for website, count in number_of_responses_per_website_expert[type].items():
        print(f"- {website}: {count}")

print("Number of experts of each type saying a credential is necessary:")
for type in types:
    print(f"Type: {type}")
    for website, credentials_dict in number_users_giving_a_credential_to_website_necessary_expert[type].items():
        print(f"- {website}:")
        for credential, count in credentials_dict.items():
            print(f"  - {credential}: {count}")


Number of expert responses per website:
Type: cybersecurity
- Shop Apotheke: 6
- Zava: 6
- Official Government Website of Germany: 6
- Stepstone: 6
- TU München: 6
- Lufthansa: 6
- FlixBus: 15
- Hertz: 15
- Sparkasse: 6
- ImmoScout24: 6
- Steam: 15
- Amazon: 15
- Facebook: 15
- Zeit: 6
- PayPal: 15
- DocMorris: 3
- Qare: 3
- Official Government Website of France: 3
- Pole Emploi: 3
- Panthéon Sorbonne: 3
- Air France: 3
- Crédit Mutuel: 3
- SeLoger: 3
- Le Monde: 3
- Redcare: 6
- SoS Pediatra: 6
- Official Government Website of Italy: 6
- Indeed: 6
- Bocconi: 6
- ITA Airways: 6
- Intesa Sanpaolo: 6
- Immobiliare: 6
- La Repubblica: 6
Type: law
- Shop Apotheke: 3
- Zava: 3
- Official Government Website of Germany: 3
- Stepstone: 3
- TU München: 3
- Lufthansa: 3
- FlixBus: 3
- Hertz: 3
- Sparkasse: 3
- ImmoScout24: 3
- Steam: 3
- Amazon: 3
- Facebook: 3
- Zeit: 3
- PayPal: 3
- DocMorris: 0
- Qare: 0
- Official Government Website of France: 0
- Pole Emploi: 0
- Panthéon Sorbonne: 0
- Air 

In [59]:
# Get number of experts giving a credential to each website category

number_of_responses_per_category_expert = {type: {name: 0 for name in categories} for type in types}
number_users_giving_a_credential_to_category_necessary_expert = {
    type: {
        category: {credential: 0 for credential in credentials}
        for category in categories}
    for type in types
}
number_users_giving_a_credential_to_category_permissible_expert = {
    type: {
        category: {credential: 0 for credential in credentials}
        for category in categories}
    for type in types
}


for type in types:
    for website, credentials_dict in number_users_giving_a_credential_to_website_necessary_expert[type].items():
        cat = website_categorization.get(website, (None, None, None))
        if not cat:
            print(f"Error website {website} not found in the categorization")
            continue
        for credential, count in credentials_dict.items():
            number_users_giving_a_credential_to_category_necessary_expert[type][cat][credential] += count
for type in types:
    for website, credentials_dict in number_users_giving_a_credential_to_website_permissible_expert[type].items():
        cat = website_categorization.get(website, (None, None, None))
        if not cat:
            print(f"Error website {website} not found in the categorization")
            continue
        for credential, count in credentials_dict.items():
            number_users_giving_a_credential_to_category_permissible_expert[type][cat][credential] += count
for type in types:
    for website, num in number_of_responses_per_website_expert[type].items():
        cat = website_categorization.get(website, (None, None, None))
        if not cat:
            print(f"Error website {website} not found in the categorization")
            continue
        number_of_responses_per_category_expert[type][cat] += num
    
print("Number of users giving a credential to a specific category:")
for type in types:
    print(f"Type: {type}")
    for category, credentials_dict in number_users_giving_a_credential_to_category_necessary_expert[type].items():
        print(f"- {category}:")
        for credential, count in credentials_dict.items():
            print(f"  - {credential}: {count}")


Number of users giving a credential to a specific category:
Type: cybersecurity
- Pharmacy:
  - Official Identity Document: 3
  - Driver's license: 12
  - Birth certificate: 8
  - Marriage certificate: 12
  - Visa: 14
  - Bank credentials: 9
  - University transcript: 14
  - Diploma: 14
  - Employment history: 14
  - Professional licenses: 13
  - Health insurance: 1
  - Prescriptions: 0
  - Medical records: 7
  - Disability status: 5
- Online Doctors:
  - Official Identity Document: 5
  - Driver's license: 12
  - Birth certificate: 9
  - Marriage certificate: 11
  - Visa: 14
  - Bank credentials: 11
  - University transcript: 14
  - Diploma: 14
  - Employment history: 13
  - Professional licenses: 14
  - Health insurance: 0
  - Prescriptions: 1
  - Medical records: 0
  - Disability status: 1
- Government:
  - Official Identity Document: 0
  - Driver's license: 1
  - Birth certificate: 1
  - Marriage certificate: 1
  - Visa: 2
  - Bank credentials: 7
  - University transcript: 7
  - Dip

In [60]:
#Percentage of experts giving a credential to each website category

percentage_users_giving_a_credential_to_category_necessary_expert = copy.deepcopy(number_users_giving_a_credential_to_category_necessary_expert)
for type in types:
    for category, credentials_dict in percentage_users_giving_a_credential_to_category_necessary_expert[type].items():
        total_responses = number_of_responses_per_category_expert[type][category]
        for credential, count in credentials_dict.items():
            if total_responses > 0:
                percentage = (count / total_responses) * 100
            else:
                percentage = 0
            percentage_users_giving_a_credential_to_category_necessary_expert[type][category][credential] = percentage
percentage_users_giving_a_credential_to_category_permissible_expert = copy.deepcopy(number_users_giving_a_credential_to_category_permissible_expert)
for type in types:
    for category, credentials_dict in percentage_users_giving_a_credential_to_category_permissible_expert[type].items():
        total_responses = number_of_responses_per_category_expert[type][category]
        for credential, count in credentials_dict.items():
            if total_responses > 0:
                percentage = (count / total_responses) * 100
            else:
                percentage = 0
            percentage_users_giving_a_credential_to_category_permissible_expert[type][category][credential] = percentage

print("Percentage of experts deeming a credential necessary to a specific category:")
for type in types:
    print(f"Type: {type}")
    for category, credentials_dict in percentage_users_giving_a_credential_to_category_necessary_expert[type].items():
        print(f"- {category}:")
        for credential, percentage in credentials_dict.items():
            print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_necessary_expert[type][category][credential]} / {number_of_responses_per_category_expert[type][category]})")

print("Percentage of experts giving an optional credential to a specific category:")
for type in types:
    print(f"Type: {type}")
    for category, credentials_dict in percentage_users_giving_a_credential_to_category_permissible_expert[type].items():
        print(f"- {category}:")
        for credential, percentage in credentials_dict.items():
            print(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_permissible_expert[type][category][credential]} / {number_of_responses_per_category_expert[type][category]})")

output_file = "data/credential_percentage_per_category_expert.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for type in types:
        if type == "law":
            f.write("\nPercentage of experts deeming a credential necessary to a specific category:\n")
        else:
            f.write("\nPercentage of experts deeming it never okay to provide a credential to a specific category:\n")
        f.write(f"\nType: {type}\n")
        for category, credentials_dict in percentage_users_giving_a_credential_to_category_necessary_expert[type].items():
            f.write(f"- {category}:\n")
            for credential, percentage in credentials_dict.items():
                f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_necessary_expert[type][category][credential]} / {number_of_responses_per_category_expert[type][category]})\n")

        f.write("\nPercentage of experts giving an optional credential to a specific category:\n")
        f.write(f"\nType: {type}\n")
        for category, credentials_dict in percentage_users_giving_a_credential_to_category_permissible_expert[type].items():
            f.write(f"- {category}:\n")
            for credential, percentage in credentials_dict.items():
                f.write(f"  - {credential}: {percentage:.2f}% ({number_users_giving_a_credential_to_category_permissible_expert[type][category][credential]} / {number_of_responses_per_category_expert[type][category]})\n")

Percentage of experts deeming a credential necessary to a specific category:
Type: cybersecurity
- Pharmacy:
  - Official Identity Document: 20.00% (3 / 15)
  - Driver's license: 80.00% (12 / 15)
  - Birth certificate: 53.33% (8 / 15)
  - Marriage certificate: 80.00% (12 / 15)
  - Visa: 93.33% (14 / 15)
  - Bank credentials: 60.00% (9 / 15)
  - University transcript: 93.33% (14 / 15)
  - Diploma: 93.33% (14 / 15)
  - Employment history: 93.33% (14 / 15)
  - Professional licenses: 86.67% (13 / 15)
  - Health insurance: 6.67% (1 / 15)
  - Prescriptions: 0.00% (0 / 15)
  - Medical records: 46.67% (7 / 15)
  - Disability status: 33.33% (5 / 15)
- Online Doctors:
  - Official Identity Document: 33.33% (5 / 15)
  - Driver's license: 80.00% (12 / 15)
  - Birth certificate: 60.00% (9 / 15)
  - Marriage certificate: 73.33% (11 / 15)
  - Visa: 93.33% (14 / 15)
  - Bank credentials: 73.33% (11 / 15)
  - University transcript: 93.33% (14 / 15)
  - Diploma: 93.33% (14 / 15)
  - Employment history: 

In [61]:
# Get number of responses per website and number of credentials given to website

output_file = "data/per_expert_output.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for id in survey_expert_cybersecurity_valid_ids + survey_expert_law_valid_ids + survey_expert_policy_valid_ids + survey_expert_ethics_valid_ids:
        answers_website = survey_expert_answer_collection.find_one({
            "id": id,
            "form": "websiteCredentialsOpinions"})
        if not answers_website:
            print(f"Error this ID should not be valid {id}")
            continue
        if id in survey_expert_cybersecurity_valid_ids:
            type = "cybersecurity"
        elif id in survey_expert_law_valid_ids:
            type = "law"
        elif id in survey_expert_policy_valid_ids:
            type = "policy"
        elif id in survey_expert_ethics_valid_ids:
            type = "ethics"
        print(f"ID: {id} of type {type}:")
        f.write(f"ID: {id} of type {type}:\n")
        for answer in answers_website['values']:
            website = answer["website"]
            print(f"For website: {website} said yes for credentials: ")
            f.write(f"For website: {website} said yes for credentials:\n")
            cat = website_categorization.get(website, (None, None, None))
            if type == "law":
                necessary = answer["necessaryCredentials"]
                permissible = answer["permissibleCredentials"]
                if necessary or permissible:
                    combined = list(set(necessary + permissible))
                    for credential_id in combined:
                        if credential_id == "None":
                            continue
                        print(f"- {credential_id}")
                        f.write(f"- {credential_id}\n")
            else:
                permissible = answer["permissibleCredentials"]
                for credential_id in permissible:
                        if credential_id == "None":
                            continue
                        print(f"- {credential_id}")
                        f.write(f"- {credential_id}\n")


ID: 39cb31cd7e8f5c15 of type cybersecurity:
For website: Shop Apotheke said yes for credentials: 
- Prescriptions
- Health insurance
- Bank credentials
For website: Zava said yes for credentials: 
- Medical records
- Disability status
- Prescriptions
- Health insurance
- Bank credentials
For website: Official Government Website of Germany said yes for credentials: 
- Official Identity Document
For website: Stepstone said yes for credentials: 
- Employment history
- Professional licenses
- Diploma
- University transcript
- Visa
- Disability status
For website: TU München said yes for credentials: 
- Official Identity Document
For website: Lufthansa said yes for credentials: 
- Official Identity Document
- Bank credentials
- Disability status
For website: FlixBus said yes for credentials: 
- Bank credentials
For website: Hertz said yes for credentials: 
- Bank credentials
For website: Sparkasse said yes for credentials: 
- Official Identity Document
- Employment history
For website: Immo

## User Study: Auxiliary Data

In [62]:
#Code for different scenario types

possible_scenarios = [
    ('all_good', 'normal', 'no', 'normal'),
    ('all_good', 'normal', 'yes', 'normal'),
    ('S1', 'G3', 'no', 'high'),
    ('S1', 'G3', 'no', 'normal'),
    ('S1', 'G4', 'no', 'normal'),
    ('S2', 'bogus', 'no', 'normal'),
    ('S2', 'vague', 'no', 'normal'),
    ('S3', 'G8', 'no', 'normal'),
    ('S3', 'G9', 'no', 'high'),
    ('S3', 'G9', 'no', 'normal'),
    ('S4', 'proper', 'no', 'normal'),
    ('S4', 'vague', 'no', 'normal'),
    ('S5', 'G11', 'no', 'normal'),
    ('S6', 'G11', 'no', 'normal'),
    ('S5', 'G12', 'no', 'normal'),
    ('S6', 'G12', 'no', 'normal'),
    ('S5', 'G13', 'no', 'normal'),
    ('S6', 'G13', 'no', 'normal'),
    ('control', 'normal', 'no', 'normal')
]

possible_scenarios_to_name = {
    ('all_good', 'normal', 'no', 'normal'): "All Good",
    ('all_good', 'normal', 'yes', 'normal'): "All Good with Expert",
    ('S1', 'G3', 'no', 'high'): "Low Confidence High Issue Rate",
    ('S1', 'G3', 'no', 'normal'): "Low Confidence",
    ('S1', 'G4', 'no', 'normal'): "High Confidence",
    ('S2', 'bogus', 'no', 'normal'): "Bogus Purpose Support System Correct",
    ('S2', 'vague', 'no', 'normal'): "Vague Purpose Support System Correct",
    ('S3', 'G8', 'no', 'normal'): "Expert Mistake",
    ('S3', 'G9', 'no', 'high'): "User Mistake High Issue Rate",
    ('S3', 'G9', 'no', 'normal'): "User Mistake",
    ('S4', 'proper', 'no', 'normal'): "Proper Purpose Support System Wrong",
    ('S4', 'vague', 'no', 'normal'): "Vague Purpose Support System Wrong",
    ('S5', 'G11', 'no', 'normal'): "User and Expert Mistake",
    ('S6', 'G11', 'no', 'normal'): "User and Expert too Conservative",
    ('S5', 'G12', 'no', 'normal'): "Expert Mistake User Correct",
    ('S6', 'G12', 'no', 'normal'): "Expert too Conservative User Correct",
    ('S5', 'G13', 'no', 'normal'): "User Mistake Expert Correct",
    ('S6', 'G13', 'no', 'normal'): "User too Conservative Expert Correct",
    ('control', 'normal', 'no', 'normal'): "Control"
}

all_good_scenario = [
    ('all_good', 'normal', 'no', 'normal'),
    ('all_good', 'normal', 'yes', 'normal'),
]

control_scenario = [('control', 'normal', 'no', 'normal')]

all_good_scenarios = [
    "s1_all_good_scenario_1", 
    "s1_all_good_scenario_2", 
    "s1_all_good_scenario_3", 
    "s1_all_good_scenario_4", 
    "s1_all_good_scenario_5", 
    "s1_all_good_scenario_6",
    "s1_all_good_scenario_7",
    "s1_all_good_scenario_8",
    "s1_all_good_scenario_9",
    "s1_all_good_scenario_10",
    "s2_all_good_scenario_1",
    "s2_all_good_scenario_2",
    "s2_all_good_scenario_3",
    "s2_all_good_scenario_4",
    "s2_all_good_scenario_5",
    "s3_all_good_expert_scenario_1",
    "s3_all_good_expert_law_scenario_1",
    "s3_all_good_expert_ethics_scenario_1",
    "s3_all_good_expert_policy_scenario_1",
    "s3_all_good_expert_cyber_scenario_1",
    "s3_all_good_user_scenario_1",
    "s3_all_good_expert_scenario_2",
    "s3_all_good_expert_law_scenario_2",
    "s3_all_good_expert_ethics_scenario_2",
    "s3_all_good_expert_policy_scenario_2",
    "s3_all_good_expert_cyber_scenario_2",
    "s3_all_good_user_scenario_2",
    "s3_all_good_expert_scenario_3",
    "s3_all_good_expert_law_scenario_3",
    "s3_all_good_expert_ethics_scenario_3",
    "s3_all_good_expert_policy_scenario_3",
    "s3_all_good_expert_cyber_scenario_3",
    "s3_all_good_user_scenario_3",
    "s3_all_good_expert_scenario_4",
    "s3_all_good_expert_law_scenario_4",
    "s3_all_good_expert_ethics_scenario_4",
    "s3_all_good_expert_policy_scenario_4",
    "s3_all_good_expert_cyber_scenario_4",
    "s3_all_good_user_scenario_4",
    "s3_all_good_expert_scenario_5",
    "s3_all_good_expert_law_scenario_5",
    "s3_all_good_expert_ethics_scenario_5",
    "s3_all_good_expert_policy_scenario_5",
    "s3_all_good_expert_cyber_scenario_5",
    "s3_all_good_user_scenario_5",
    "s3_all_good_expert_scenario_6",
    "s3_all_good_expert_law_scenario_6",
    "s3_all_good_expert_ethics_scenario_6",
    "s3_all_good_expert_policy_scenario_6",
    "s3_all_good_expert_cyber_scenario_6",
    "s3_all_good_user_scenario_6",
    "s3_all_good_expert_scenario_7",
    "s3_all_good_expert_law_scenario_7",
    "s3_all_good_expert_ethics_scenario_7",
    "s3_all_good_expert_policy_scenario_7",
    "s3_all_good_expert_cyber_scenario_7",
    "s3_all_good_user_scenario_7",
    "s3_all_good_expert_scenario_8",
    "s3_all_good_expert_law_scenario_8",
    "s3_all_good_expert_ethics_scenario_8",
    "s3_all_good_expert_policy_scenario_8",
    "s3_all_good_expert_cyber_scenario_8",
    "s3_all_good_user_scenario_8",
    "s3_all_good_expert_scenario_9",
    "s3_all_good_expert_law_scenario_9",
    "s3_all_good_expert_ethics_scenario_9",
    "s3_all_good_expert_policy_scenario_9",
    "s3_all_good_expert_cyber_scenario_9",
    "s3_all_good_user_scenario_9",
    "s3_all_good_expert_scenario_10",
    "s3_all_good_expert_law_scenario_10",
    "s3_all_good_expert_ethics_scenario_10",
    "s3_all_good_expert_policy_scenario_10",
    "s3_all_good_expert_cyber_scenario_10",
    "s3_all_good_user_scenario_10",
    "s4_all_good_no_purpose_scenario_1",
    "s4_all_good_purpose_proper_scenario_1",
    "s4_all_good_purpose_vague_scenario_1",
    "s4_all_good_no_purpose_scenario_2",
    "s4_all_good_purpose_proper_scenario_2",
    "s4_all_good_purpose_vague_scenario_2",
    "s4_all_good_no_purpose_scenario_3",
    "s4_all_good_purpose_proper_scenario_3",
    "s4_all_good_purpose_vague_scenario_3",
    "s4_all_good_no_purpose_scenario_4",
    "s4_all_good_purpose_proper_scenario_4",
    "s4_all_good_purpose_vague_scenario_4",
    "s4_all_good_no_purpose_scenario_5",
    "s4_all_good_purpose_proper_scenario_5",
    "s4_all_good_purpose_vague_scenario_5",
    "s5_all_good_scenario_1",
    "s5_all_good_law_scenario_1",
    "s5_all_good_ethics_scenario_1",
    "s5_all_good_policy_scenario_1",
    "s5_all_good_cyber_scenario_1",
    "s5_all_good_scenario_2",
    "s5_all_good_law_scenario_2",
    "s5_all_good_ethics_scenario_2",
    "s5_all_good_policy_scenario_2",
    "s5_all_good_cyber_scenario_2",
    "s5_all_good_scenario_3",
    "s5_all_good_law_scenario_3",
    "s5_all_good_ethics_scenario_3",
    "s5_all_good_policy_scenario_3",
    "s5_all_good_cyber_scenario_3",
    "s5_all_good_scenario_4",
    "s5_all_good_law_scenario_4",
    "s5_all_good_ethics_scenario_4",
    "s5_all_good_policy_scenario_4",
    "s5_all_good_cyber_scenario_4",
    "s5_all_good_scenario_5",
    "s5_all_good_law_scenario_5",
    "s5_all_good_ethics_scenario_5",
    "s5_all_good_policy_scenario_5",
    "s5_all_good_cyber_scenario_5",
    "s5_all_good_scenario_6",
    "s5_all_good_law_scenario_6",
    "s5_all_good_ethics_scenario_6",
    "s5_all_good_policy_scenario_6",
    "s5_all_good_cyber_scenario_6",
    "s5_all_good_scenario_7",
    "s5_all_good_law_scenario_7",
    "s5_all_good_ethics_scenario_7",
    "s5_all_good_policy_scenario_7",
    "s5_all_good_cyber_scenario_7",
    "s5_all_good_scenario_8",
    "s5_all_good_law_scenario_8",
    "s5_all_good_ethics_scenario_8",
    "s5_all_good_policy_scenario_8",
    "s5_all_good_cyber_scenario_8",
    "s5_all_good_scenario_9",
    "s5_all_good_law_scenario_9",
    "s5_all_good_ethics_scenario_9",
    "s5_all_good_policy_scenario_9",
    "s5_all_good_cyber_scenario_9",
    "s5_all_good_scenario_10",
    "s5_all_good_law_scenario_10",
    "s5_all_good_ethics_scenario_10",
    "s5_all_good_policy_scenario_10",
    "s5_all_good_cyber_scenario_10",
    "s6_all_good_scenario_1",
    "s6_all_good_law_scenario_1",
    "s6_all_good_ethics_scenario_1",
    "s6_all_good_policy_scenario_1",
    "s6_all_good_cyber_scenario_1",
    "s6_all_good_scenario_2",
    "s6_all_good_law_scenario_2",
    "s6_all_good_ethics_scenario_2",
    "s6_all_good_policy_scenario_2",
    "s6_all_good_cyber_scenario_2",
    "s6_all_good_scenario_3",
    "s6_all_good_law_scenario_3",
    "s6_all_good_ethics_scenario_3",
    "s6_all_good_policy_scenario_3",
    "s6_all_good_cyber_scenario_3",
    "s6_all_good_scenario_4",
    "s6_all_good_law_scenario_4",
    "s6_all_good_ethics_scenario_4",
    "s6_all_good_policy_scenario_4",
    "s6_all_good_cyber_scenario_4",
    "s6_all_good_scenario_5",
    "s6_all_good_law_scenario_5",
    "s6_all_good_ethics_scenario_5",
    "s6_all_good_policy_scenario_5",
    "s6_all_good_cyber_scenario_5",
    "s6_all_good_scenario_6",
    "s6_all_good_law_scenario_6",
    "s6_all_good_ethics_scenario_6",
    "s6_all_good_policy_scenario_6",
    "s6_all_good_cyber_scenario_6",
    "s6_all_good_scenario_7",
    "s6_all_good_law_scenario_7",
    "s6_all_good_ethics_scenario_7",
    "s6_all_good_policy_scenario_7",
    "s6_all_good_cyber_scenario_7",
    "s6_all_good_scenario_8",
    "s6_all_good_law_scenario_8",
    "s6_all_good_ethics_scenario_8",
    "s6_all_good_policy_scenario_8",
    "s6_all_good_cyber_scenario_8",
    "s6_all_good_scenario_9",
    "s6_all_good_law_scenario_9",
    "s6_all_good_ethics_scenario_9",
    "s6_all_good_policy_scenario_9",
    "s6_all_good_cyber_scenario_9",
    "s6_all_good_scenario_10",
    "s6_all_good_law_scenario_10",
    "s6_all_good_ethics_scenario_10",
    "s6_all_good_policy_scenario_10",
    "s6_all_good_cyber_scenario_10",
]
with open("files/user_study_scenario_id_mapping.json", "r", encoding="utf-8") as f:
    equivalent_scenarios = json.load(f)

scenarios_to_aggregate = {
    "S1": ["S1_1", "S1_2", "S1_3", "S1_4", "S1_5", "S1_6", "S1_7", "S1_8", "S1_9", "S1_10"],
    "S2": ["S2_1", "S2_2", "S2_3", "S2_4", "S2_5"],
    "S3": ["S3_1", "S3_2", "S3_3", "S3_4", "S3_5", "S3_6", "S3_7", "S3_8", "S3_9", "S3_10"],
    "S4": ["S4_1", "S4_2", "S4_3", "S4_4", "S4_5"],
    "S5": ["S5_1", "S5_2", "S5_3", "S5_4", "S5_5", "S5_6", "S5_7", "S5_8", "S5_9", "S5_10"],
    "S6": ["S6_1", "S6_2", "S6_3", "S6_4", "S6_5", "S6_6", "S6_7", "S6_8", "S6_9", "S6_10"],
}

with open("files/user_study_website_data.json", "r", encoding="utf-8") as f:
    scenario_to_website_data = json.load(f)

map_id_to_scenario = {}
for scenario, data in equivalent_scenarios.items():
    for scenario_id in data['all_good'] + data['control'] + data['test']:
        map_id_to_scenario[scenario_id] = scenario

with open("files/user_study_scenario_ids_with_expert_type.json", "r", encoding="utf-8") as f:
    expert_type_scenarios = json.load(f)

## User Study: Generate unaggregated Scenario result outputs

In [63]:
scenario_responses = {}
scenario_responses_all_good_in_group = {}
scenario_responses_all_good_overall = {}
scenario_responses_control = {}

for curr_id in user_study_valid_ids:
    answers_scenarios = user_study_answer_collection.find_one({
        "id": curr_id,
        "form": "scenarioAnswers"
    })
    if not answers_scenarios:
        continue
    participant_group = (
        answers_scenarios['scenarioGroup'],
        answers_scenarios['group'],
        answers_scenarios['expertType'],
        answers_scenarios['badResponseRate']
    )
    participant_group_name = possible_scenarios_to_name.get(participant_group, "Unknown Scenario")
    if participant_group_name == "Control": 
        for answer in answers_scenarios['values']:
            scenario  = answer['website']
            response = answer['decision']
            if scenario not in scenario_responses_control:
                scenario_responses_control[scenario] = {'yes': 0, 'no': 0}
            scenario_responses_control[scenario][response] += 1
        continue
    if participant_group_name == "All Good" or participant_group_name == "All Good with Expert":
        for answer in answers_scenarios['values']:
            scenario  = answer['website']
            response = answer['decision']
            if scenario not in scenario_responses_all_good_in_group:
                scenario_responses_all_good_in_group[scenario] = {'yes': 0, 'no': 0}
            scenario_responses_all_good_in_group[scenario][response] += 1
            if scenario not in scenario_responses_all_good_overall:
                scenario_responses_all_good_overall[scenario] = {'yes': 0, 'no': 0}
            scenario_responses_all_good_overall[scenario][response] += 1
        continue
    for answer in answers_scenarios['values']:
        scenario  = answer['website']
        response = answer['decision']
        if scenario in all_good_scenarios: 
            if scenario not in scenario_responses_all_good_overall:
                scenario_responses_all_good_overall[scenario] = {'yes': 0, 'no': 0}
            scenario_responses_all_good_overall[scenario][response] += 1
            continue
        if scenario not in scenario_responses:
            scenario_responses[scenario] = {'yes': 0, 'no': 0}
        scenario_responses[scenario][response] += 1

print("Aggregated regardless of group\n")

for scenario, responses in scenario_responses.items():
    total = responses['yes'] + responses['no']
    yes_percentage = (responses['yes'] / total) * 100 if total > 0 else 0
    no_percentage = (responses['no'] / total) * 100 if total > 0 else 0
    print(f"Scenario: {scenario}")
    print(f"  Yes: {responses['yes']} ({yes_percentage:.2f}%)")
    print(f"  No: {responses['no']} ({no_percentage:.2f}%)")
    print(f"  Total: {total}\n")

Aggregated regardless of group

Scenario: s5_g13_scenario_9
  Yes: 0 (0.00%)
  No: 4 (100.00%)
  Total: 4

Scenario: s5_g13_scenario_7
  Yes: 3 (20.00%)
  No: 12 (80.00%)
  Total: 15

Scenario: s5_g12_scenario_1
  Yes: 2 (12.50%)
  No: 14 (87.50%)
  Total: 16

Scenario: s5_g12_scenario_9
  Yes: 2 (15.38%)
  No: 11 (84.62%)
  Total: 13

Scenario: s1_g4_scenario_2
  Yes: 8 (61.54%)
  No: 5 (38.46%)
  Total: 13

Scenario: s1_g4_scenario_9
  Yes: 14 (100.00%)
  No: 0 (0.00%)
  Total: 14

Scenario: s6_g13_scenario_4
  Yes: 12 (92.31%)
  No: 1 (7.69%)
  Total: 13

Scenario: s6_g13_scenario_7
  Yes: 7 (63.64%)
  No: 4 (36.36%)
  Total: 11

Scenario: s1_g3_scenario_4
  Yes: 16 (30.77%)
  No: 36 (69.23%)
  Total: 52

Scenario: s1_g3_scenario_6
  Yes: 21 (56.76%)
  No: 16 (43.24%)
  Total: 37

Scenario: s1_g3_scenario_10
  Yes: 39 (78.00%)
  No: 11 (22.00%)
  Total: 50

Scenario: s1_g3_scenario_7
  Yes: 20 (37.74%)
  No: 33 (62.26%)
  Total: 53

Scenario: s3_g9_scenario_1
  Yes: 0 (0.00%)
  No: 

In [64]:
output_file = "data/user_study_results_no_agg.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Results for equivalent scenarios\n")
    print("Results for equivalent scenarios\n")
    for group, types in equivalent_scenarios.items():
        print(f"Group {group}\n")
        print(f"All Good Scenario:")
        f.write(f"\nGroup {group}\n")
        f.write(f"All Good Scenario:\n")
        for all_good_scenario in types['all_good']:
            if all_good_scenario in scenario_responses_all_good_in_group:
                total = scenario_responses_all_good_in_group[all_good_scenario]['yes'] + scenario_responses_all_good_in_group[all_good_scenario]['no']
                yes_percentage = (scenario_responses_all_good_in_group[all_good_scenario]['yes'] / total) * 100 if total > 0 else 0
                no_percentage = (scenario_responses_all_good_in_group[all_good_scenario]['no'] / total) * 100 if total > 0 else 0
                print(f"  Scenario: {all_good_scenario}")
                print(f"  Yes: {scenario_responses_all_good_in_group[all_good_scenario]['yes']} ({yes_percentage:.2f}%)")
                print(f"  No: {scenario_responses_all_good_in_group[all_good_scenario]['no']} ({no_percentage:.2f}%)")
                print(f"  Total: {total}\n")

                f.write(f"  Scenario: {all_good_scenario}\n")
                f.write(f"  Yes: {scenario_responses_all_good_in_group[all_good_scenario]['yes']} ({yes_percentage:.2f}%)\n")
                f.write(f"  No: {scenario_responses_all_good_in_group[all_good_scenario]['no']} ({no_percentage:.2f}%)\n")
                f.write(f"  Total: {total}\n")
            if all_good_scenario in scenario_responses_all_good_overall:
                total = scenario_responses_all_good_overall[all_good_scenario]['yes'] + scenario_responses_all_good_overall[all_good_scenario]['no']
                yes_percentage = (scenario_responses_all_good_overall[all_good_scenario]['yes'] / total) * 100 if total > 0 else 0
                no_percentage = (scenario_responses_all_good_overall[all_good_scenario]['no'] / total) * 100 if total > 0 else 0
                print(f"  Scenario Overall: {all_good_scenario}")
                print(f"  Yes: {scenario_responses_all_good_overall[all_good_scenario]['yes']} ({yes_percentage:.2f}%)")
                print(f"  No: {scenario_responses_all_good_overall[all_good_scenario]['no']} ({no_percentage:.2f}%)")
                print(f"  Total: {total}\n")

                f.write(f"  Scenario Overall: {all_good_scenario}\n")
                f.write(f"  Yes: {scenario_responses_all_good_overall[all_good_scenario]['yes']} ({yes_percentage:.2f}%)\n")
                f.write(f"  No: {scenario_responses_all_good_overall[all_good_scenario]['no']} ({no_percentage:.2f}%)\n")
                f.write(f"  Total: {total}\n")
        for control_scenario in types['control']:
            if control_scenario in scenario_responses_control:
                total = scenario_responses_control[control_scenario]['yes'] + scenario_responses_control[control_scenario]['no']
                yes_percentage = (scenario_responses_control[control_scenario]['yes'] / total) * 100 if total > 0 else 0
                no_percentage = (scenario_responses_control[control_scenario]['no'] / total) * 100 if total > 0 else 0
                print(f"Control Scenario: {control_scenario}")
                print(f"  Yes: {scenario_responses_control[control_scenario]['yes']} ({yes_percentage:.2f}%)")
                print(f"  No: {scenario_responses_control[control_scenario]['no']} ({no_percentage:.2f}%)")
                print(f"  Total: {total}\n")

                f.write(f"Control Scenario: {control_scenario}\n")
                f.write(f"  Yes: {scenario_responses_control[control_scenario]['yes']} ({yes_percentage:.2f}%)\n")
                f.write(f"  No: {scenario_responses_control[control_scenario]['no']} ({no_percentage:.2f}%)\n")
                f.write(f"  Total: {total}\n")
        for test_scenario in types['test']:
            if test_scenario in scenario_responses:
                total = scenario_responses[test_scenario]['yes'] + scenario_responses[test_scenario]['no']
                yes_percentage = (scenario_responses[test_scenario]['yes'] / total) * 100 if total > 0 else 0
                no_percentage = (scenario_responses[test_scenario]['no'] / total) * 100 if total > 0 else 0
                print(f"Test Scenario: {test_scenario}")
                print(f"  Yes: {scenario_responses[test_scenario]['yes']} ({yes_percentage:.2f}%)")
                print(f"  No: {scenario_responses[test_scenario]['no']} ({no_percentage:.2f}%)")
                print(f"  Total: {total}\n")

                f.write(f"Test Scenario: {test_scenario}\n")
                f.write(f"  Yes: {scenario_responses[test_scenario]['yes']} ({yes_percentage:.2f}%)\n")
                f.write(f"  No: {scenario_responses[test_scenario]['no']} ({no_percentage:.2f}%)\n")
                f.write(f"  Total: {total}\n")

Results for equivalent scenarios

Group S1_1

All Good Scenario:
  Scenario: s1_all_good_scenario_1
  Yes: 25 (92.59%)
  No: 2 (7.41%)
  Total: 27

  Scenario Overall: s1_all_good_scenario_1
  Yes: 238 (88.48%)
  No: 31 (11.52%)
  Total: 269

Control Scenario: s1_all_good_scenario_1
  Yes: 11 (84.62%)
  No: 2 (15.38%)
  Total: 13

Test Scenario: s1_g3_scenario_1
  Yes: 47 (87.04%)
  No: 7 (12.96%)
  Total: 54

Test Scenario: s1_g4_scenario_1
  Yes: 19 (95.00%)
  No: 1 (5.00%)
  Total: 20

Group S1_2

All Good Scenario:
  Scenario: s1_all_good_scenario_2
  Yes: 10 (52.63%)
  No: 9 (47.37%)
  Total: 19

  Scenario Overall: s1_all_good_scenario_2
  Yes: 136 (54.18%)
  No: 115 (45.82%)
  Total: 251

Control Scenario: s1_all_good_scenario_2
  Yes: 5 (38.46%)
  No: 8 (61.54%)
  Total: 13

Test Scenario: s1_g3_scenario_2
  Yes: 16 (29.63%)
  No: 38 (70.37%)
  Total: 54

Test Scenario: s1_g4_scenario_2
  Yes: 8 (61.54%)
  No: 5 (38.46%)
  Total: 13

Group S1_3

All Good Scenario:
  Scenario: s

## User Study: Data Aggregating results by Scenario Type

In [65]:
#Output results aggregated by scenario type

output_file = "data/user_study_results_agg.txt"

with open(output_file, "w", encoding="utf-8") as f:

    print("Aggregated Results for Equivalent Scenarios\n")
    f.write("Aggregated Results for Equivalent Scenarios\n")

    for main_group, subgroups in scenarios_to_aggregate.items():
        print(f"\n=== Group {main_group} ===\n")
        f.write(f"\n=== Group {main_group} ===\n")

        valid_subgroups = [g for g in subgroups if g in equivalent_scenarios]
        if not valid_subgroups:
            continue

        reference = equivalent_scenarios[valid_subgroups[0]]

        for scenario_type, scenario_list in reference.items():
            print(f"{scenario_type.upper()} scenarios:\n")
            f.write(f"{scenario_type.upper()} scenarios:\n\n")
            for i in range(len(scenario_list)):
                aggregated_yes = 0
                aggregated_no = 0
                total = 0
                scenario_names = []
                aggregated_yes_overall = 0
                aggregated_no_overall = 0
                total_overall = 0

                for subgroup in valid_subgroups:
                    subgroup_data = equivalent_scenarios[subgroup]
                    if scenario_type in subgroup_data and i < len(subgroup_data[scenario_type]):
                        scenario_name = subgroup_data[scenario_type][i]
                        scenario_names.append(scenario_name)

                        if scenario_type == "all_good":
                            responses = scenario_responses_all_good_in_group.get(scenario_name)
                            responses_overall = scenario_responses_all_good_overall.get(scenario_name)
                        elif scenario_type == "control":
                            responses = scenario_responses_control.get(scenario_name)
                        else:  
                            responses = scenario_responses.get(scenario_name)

                        if responses:
                            yes = responses["yes"]
                            no = responses["no"]
                            aggregated_yes += yes
                            aggregated_no += no
                            total += yes + no
                        if responses_overall:
                            yes_overall = responses_overall["yes"]
                            no_overall = responses_overall["no"]
                            aggregated_yes_overall += yes_overall
                            aggregated_no_overall += no_overall
                            total_overall += yes_overall + no_overall

                if total > 0:
                    yes_pct = (aggregated_yes / total) * 100
                    no_pct = (aggregated_no / total) * 100
                    print(f"  Aggregated for: {', '.join(scenario_names)}")
                    print(f"    Yes: {aggregated_yes} ({yes_pct:.2f}%)")
                    print(f"    No: {aggregated_no} ({no_pct:.2f}%)")
                    print(f"    Total: {total}\n")

                    f.write(f"  Aggregated for: {', '.join(scenario_names)}\n")
                    f.write(f"    Yes: {aggregated_yes} ({yes_pct:.2f}%)\n")
                    f.write(f"    No: {aggregated_no} ({no_pct:.2f}%)\n\n")
                if total_overall > 0 and scenario_type == "all_good":
                    yes_pct_overall = (aggregated_yes_overall / total_overall) * 100
                    no_pct_overall = (aggregated_no_overall / total_overall) * 100
                    print(f"  Overall Aggregated for: {', '.join(scenario_names)}")
                    print(f"    Yes: {aggregated_yes_overall} ({yes_pct_overall:.2f}%)")
                    print(f"    No: {aggregated_no_overall} ({no_pct_overall:.2f}%)")
                    print(f"    Total: {total_overall}\n")

                    f.write(f"  Overall Aggregated for: {', '.join(scenario_names)}\n")
                    f.write(f"    Yes: {aggregated_yes_overall} ({yes_pct_overall:.2f}%)\n")
                    f.write(f"    No: {aggregated_no_overall} ({no_pct_overall:.2f}%)\n\n")

Aggregated Results for Equivalent Scenarios


=== Group S1 ===

ALL_GOOD scenarios:

  Aggregated for: s1_all_good_scenario_1, s1_all_good_scenario_2, s1_all_good_scenario_3, s1_all_good_scenario_4, s1_all_good_scenario_5, s1_all_good_scenario_6, s1_all_good_scenario_7, s1_all_good_scenario_8, s1_all_good_scenario_9, s1_all_good_scenario_10
    Yes: 176 (75.86%)
    No: 56 (24.14%)
    Total: 232

  Overall Aggregated for: s1_all_good_scenario_1, s1_all_good_scenario_2, s1_all_good_scenario_3, s1_all_good_scenario_4, s1_all_good_scenario_5, s1_all_good_scenario_6, s1_all_good_scenario_7, s1_all_good_scenario_8, s1_all_good_scenario_9, s1_all_good_scenario_10
    Yes: 1826 (74.05%)
    No: 640 (25.95%)
    Total: 2466

CONTROL scenarios:

  Aggregated for: s1_all_good_scenario_1, s1_all_good_scenario_2, s1_all_good_scenario_3, s1_all_good_scenario_4, s1_all_good_scenario_5, s1_all_good_scenario_6, s1_all_good_scenario_7, s1_all_good_scenario_8, s1_all_good_scenario_9, s1_all_good_scenar

## User Study: Output Aggregated results for the control group

In [66]:
# Control for each scenario

output_file = "data/user_study_control_results_no_agg.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Results for equivalent scenarios\n")
    print("Results for equivalent scenarios\n")
    for group, types in equivalent_scenarios.items():
        print(f"Group {group}\n")
        print(f"All Good Scenario:")
        f.write(f"\nGroup {group}\n")
        f.write(f"All Good Scenario:\n")
        for control_scenario in types['control']:
            if control_scenario in scenario_responses_control:
                if control_scenario in expert_type_scenarios:
                    continue
                total = scenario_responses_control[control_scenario]['yes'] + scenario_responses_control[control_scenario]['no']
                yes_percentage = (scenario_responses_control[control_scenario]['yes'] / total) * 100 if total > 0 else 0
                no_percentage = (scenario_responses_control[control_scenario]['no'] / total) * 100 if total > 0 else 0
                print(f"Control Scenario: {control_scenario}")
                print(f"  Yes: {scenario_responses_control[control_scenario]['yes']} ({yes_percentage:.2f}%)")
                print(f"  No: {scenario_responses_control[control_scenario]['no']} ({no_percentage:.2f}%)")
                print(f"  Total: {total}\n")

                f.write(f"Control Scenario: {control_scenario}\n")
                f.write(f"  Yes: {scenario_responses_control[control_scenario]['yes']} ({yes_percentage:.2f}%)\n")
                f.write(f"  No: {scenario_responses_control[control_scenario]['no']} ({no_percentage:.2f}%)\n")
                f.write(f"  Total: {total}\n")

Results for equivalent scenarios

Group S1_1

All Good Scenario:
Control Scenario: s1_all_good_scenario_1
  Yes: 11 (84.62%)
  No: 2 (15.38%)
  Total: 13

Group S1_2

All Good Scenario:
Control Scenario: s1_all_good_scenario_2
  Yes: 5 (38.46%)
  No: 8 (61.54%)
  Total: 13

Group S1_3

All Good Scenario:
Control Scenario: s1_all_good_scenario_3
  Yes: 9 (52.94%)
  No: 8 (47.06%)
  Total: 17

Group S1_4

All Good Scenario:
Control Scenario: s1_all_good_scenario_4
  Yes: 5 (41.67%)
  No: 7 (58.33%)
  Total: 12

Group S1_5

All Good Scenario:
Control Scenario: s1_all_good_scenario_5
  Yes: 2 (11.11%)
  No: 16 (88.89%)
  Total: 18

Group S1_6

All Good Scenario:
Control Scenario: s1_all_good_scenario_6
  Yes: 10 (90.91%)
  No: 1 (9.09%)
  Total: 11

Group S1_7

All Good Scenario:
Control Scenario: s1_all_good_scenario_7
  Yes: 10 (47.62%)
  No: 11 (52.38%)
  Total: 21

Group S1_8

All Good Scenario:
Control Scenario: s1_all_good_scenario_8
  Yes: 15 (100.00%)
  No: 0 (0.00%)
  Total: 15

G